# PDCP · N01 — PDSG core tests and stress-range diagnostic
**Project:** Proxy-Decoupled Counterfactual Probing (PDCP). **Notebook 1 of 13.**

**Kaggle settings:** Accelerator **None (CPU)** · Internet **On** (only used if a package is missing) · Persistence: Files only.
Run with **Save Version → Save & Run All (Commit)**. Expected runtime: about 20–30 min on Kaggle CPU.

**What this notebook does**
1. Writes the shared module `pdcp_utils.py` (metrics, surrogates, PDSG optimiser, realism checks, renderer, parser, bootstrap).
2. Tests it: toy cases, Kruskal stress vs scikit-learn, surrogate–exact agreement, gradient checks, parser, bootstrap.
3. Optimiser convergence sweep over all graph families and sizes.
4. **Stress-range diagnostic:** for each family, which stress values can be reached by *natural-looking* drawings
   (node uniformity held at its Kamada–Kawai level, realism checks passing). This decides the stress range used in N02/N03.

**Outputs (in `/kaggle/working/n01_output/`)** — pass this folder to N02 via *Add Input → Notebook output*:
`pdcp_utils.py`, `n01_report.json`, `convergence.csv`, `stress_range.csv`, `fig_*.png`, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS (the only cell you ever need to edit) ====
OUTPUT_DIR = "/kaggle/working/n01_output"
SEED = 2026
FAST = False          # True = quick smoke test (~3 min); False = full run
N_CPU_THREADS = 4     # Kaggle CPU sessions have 4 cores

In [2]:
import os, sys, json, time, platform, subprocess, importlib
os.makedirs(OUTPUT_DIR, exist_ok=True)
def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=True)
for p, pn in [("networkx", None), ("sklearn", "scikit-learn"), ("scipy", None), ("PIL", "pillow"),
              ("torch", None), ("pandas", None), ("matplotlib", None)]:
    ensure(p, pn)
import numpy as np, pandas as pd, networkx as nx, torch, sklearn, scipy, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.set_num_threads(N_CPU_THREADS)
ENV = {"python": platform.python_version(), "numpy": np.__version__, "networkx": nx.__version__,
       "torch": torch.__version__, "sklearn": sklearn.__version__, "scipy": scipy.__version__}
print(ENV)
T_START = time.time()

{'python': '3.13.18', 'numpy': '2.1.3', 'networkx': '3.7', 'torch': '2.11.0+cpu', 'sklearn': '1.6.1', 'scipy': '1.16.3'}


## 1. Write the shared module `pdcp_utils.py`
Every later notebook imports this file from N01's output. Do not edit it here; changes go into a new version.

In [3]:
%%writefile /kaggle/working/n01_output/pdcp_utils.py
"""
pdcp_utils.py  —  shared code for the PDCP project (Proxy-Decoupled Counterfactual Probing)
Version: 0.1.0  (written by N01, 2026-11)

Contents
  1. Graph families            make_graph(family, n, seed)
  2. Exact metrics             ksm, node_uniformity, edge_length_deviation, crossing_ratio, all_metrics
  3. Differentiable surrogates (torch)  used only inside the optimiser
  4. PDSG Stage 1              pdsg_optimise(G, targets, ...)
  5. Rendering                 render_png(G, X, path, ...)
  6. Parsing model output      parse_answer(text)
  7. Statistics helpers        bootstrap_ci(...)

All coordinates X are numpy arrays of shape (n, 2). Node order = sorted(G.nodes()) after relabel to 0..n-1.
"""
from __future__ import annotations

import json
import math
import re
import time
from dataclasses import dataclass, field

import networkx as nx
import numpy as np

__version__ = "0.1.0"

# ----------------------------------------------------------------------------
# 1. Graph families
# ----------------------------------------------------------------------------
FAMILY_SIZES = {
    "ER": [10, 25, 50, 100],
    "BA": [25, 50, 100],
    "SBM": [25, 50, 100],
    "WS": [25, 50],
    "GRID": [16, 25, 49, 100],
    "TREE": [25, 50, 100],
}


def _connected_or_retry(fn, seed, max_tries=200):
    for t in range(max_tries):
        G = fn(seed + 1000 * t)
        if nx.is_connected(G):
            return G
    raise RuntimeError("could not sample a connected graph")


def make_graph(family: str, n: int, seed: int) -> nx.Graph:
    """Return a connected simple graph with nodes relabelled 0..n-1."""
    family = family.upper()
    if family == "ER":
        p = 3.5 / (n - 1)  # expected average degree 3.5
        G = _connected_or_retry(lambda s: nx.gnp_random_graph(n, p, seed=s), seed)
    elif family == "BA":
        G = nx.barabasi_albert_graph(n, 2, seed=seed)
    elif family == "SBM":
        k = {25: 2, 50: 3, 100: 4}.get(n, max(2, n // 25))
        p_in = {25: 0.40, 50: 0.25, 100: 0.15}.get(n, 0.2)
        sizes = [n // k] * k
        sizes[-1] += n - sum(sizes)
        P = [[p_in if i == j else 0.02 for j in range(k)] for i in range(k)]
        G = _connected_or_retry(lambda s: nx.stochastic_block_model(sizes, P, seed=s), seed)
        G = nx.Graph(G)
    elif family == "WS":
        G = nx.connected_watts_strogatz_graph(n, 4, 0.1, seed=seed)
    elif family == "GRID":
        a = int(round(math.sqrt(n)))
        G = nx.grid_2d_graph(a, a)
    elif family == "TREE":
        try:
            G = nx.random_labeled_tree(n, seed=seed)
        except AttributeError:  # older networkx
            G = nx.random_tree(n, seed=seed)
    else:
        raise ValueError(f"unknown family {family}")
    G = nx.convert_node_labels_to_integers(G, ordering="sorted")
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


def graph_arrays(G: nx.Graph):
    """Edge index array (m,2), shortest-path matrix D (n,n), and index arrays for i<j pairs."""
    n = G.number_of_nodes()
    E = np.array(sorted((min(u, v), max(u, v)) for u, v in G.edges()), dtype=np.int64)
    D = np.zeros((n, n), dtype=np.float64)
    for s, lengths in nx.all_pairs_shortest_path_length(G):
        for t, d in lengths.items():
            D[s, t] = d
    iu, ju = np.triu_indices(n, k=1)
    return E, D, iu, ju


def nonadjacent_edge_pairs(E: np.ndarray):
    """All pairs of edges that share no endpoint (the pairs that can cross)."""
    m = len(E)
    a, b = np.triu_indices(m, k=1)
    e1, e2 = E[a], E[b]
    share = (e1[:, 0] == e2[:, 0]) | (e1[:, 0] == e2[:, 1]) | (e1[:, 1] == e2[:, 0]) | (e1[:, 1] == e2[:, 1])
    return a[~share], b[~share]


# ----------------------------------------------------------------------------
# 2. Exact metrics
# ----------------------------------------------------------------------------
def normalise(X: np.ndarray) -> np.ndarray:
    """Uniform scale into the unit box, aspect ratio preserved, centred."""
    X = np.asarray(X, dtype=np.float64)
    X = X - X.min(0)
    s = X.max()
    if s <= 0:
        return np.full_like(X, 0.5)
    X = X / s
    X = X + (1.0 - X.max(0)) / 2.0
    return X


def _pav(y: np.ndarray, w: np.ndarray | None = None) -> np.ndarray:
    """Pool-adjacent-violators: isotonic (non-decreasing) fit of y in given order."""
    y = np.asarray(y, dtype=np.float64)
    w = np.ones_like(y) if w is None else np.asarray(w, dtype=np.float64)
    vals, wts, cnt = [], [], []
    for yi, wi in zip(y, w):
        vals.append(yi); wts.append(wi); cnt.append(1)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            v2, w2, c2 = vals.pop(), wts.pop(), cnt.pop()
            v1, w1, c1 = vals.pop(), wts.pop(), cnt.pop()
            wt = w1 + w2
            vals.append((v1 * w1 + v2 * w2) / wt); wts.append(wt); cnt.append(c1 + c2)
    return np.repeat(vals, cnt)


def ksm(X: np.ndarray, D: np.ndarray, iu=None, ju=None, ties: str = "secondary") -> float:
    """
    Kruskal (non-metric) stress in [0,1]; dhat = isotonic regression of e on d.
    ties='secondary' (default): all pairs with equal graph distance get the same dhat
        -> identical to sklearn.isotonic.IsotonicRegression, the convention used by Mooney et al.
    ties='primary': Kruskal's primary approach (ties may be broken); gives lower values. Kept for reference only.
    """
    n = len(X)
    if iu is None:
        iu, ju = np.triu_indices(n, k=1)
    d = D[iu, ju]
    e = np.linalg.norm(X[iu] - X[ju], axis=1)
    if ties == "primary":
        order = np.lexsort((e, d))
        dhat = np.empty_like(e)
        dhat[order] = _pav(e[order])
    else:
        levels, inv, cnt = np.unique(d, return_inverse=True, return_counts=True)
        means = np.bincount(inv, weights=e) / cnt
        fit = _pav(means, cnt)
        dhat = fit[inv]
    num = np.sum((e - dhat) ** 2)
    den = np.sum(e ** 2)
    return float(np.sqrt(num / den)) if den > 0 else 0.0


def node_uniformity(X: np.ndarray) -> float:
    """0 = nodes spread uniformly over a ceil(sqrt n)^2 grid on the drawing's bounding box, 1 = all in one cell."""
    n = len(X)
    g = int(math.ceil(math.sqrt(n)))
    lo, hi = X.min(0), X.max(0)
    span = np.where(hi - lo > 0, hi - lo, 1.0)
    U = (X - lo) / span
    idx = np.minimum((U * g).astype(int), g - 1)
    counts = np.zeros((g, g))
    np.add.at(counts, (idx[:, 0], idx[:, 1]), 1)
    cbar = n / g ** 2
    num = np.sqrt(np.sum((counts - cbar) ** 2))
    den = np.sqrt((n - cbar) ** 2 + (g ** 2 - 1) * cbar ** 2)
    return float(num / den)


def edge_length_deviation(X: np.ndarray, E: np.ndarray) -> float:
    """Coefficient of variation of edge lengths (scale invariant)."""
    L = np.linalg.norm(X[E[:, 0]] - X[E[:, 1]], axis=1)
    mu = L.mean()
    return float(L.std() / mu) if mu > 0 else 0.0


def _orient(p, q, r):
    return (q[..., 0] - p[..., 0]) * (r[..., 1] - p[..., 1]) - (q[..., 1] - p[..., 1]) * (r[..., 0] - p[..., 0])


def crossing_count(X: np.ndarray, E: np.ndarray, pairs=None) -> int:
    if pairs is None:
        pairs = nonadjacent_edge_pairs(E)
    a, b = pairs
    if len(a) == 0:
        return 0
    P1, P2 = X[E[a, 0]], X[E[a, 1]]
    Q1, Q2 = X[E[b, 0]], X[E[b, 1]]
    o1, o2 = _orient(P1, P2, Q1), _orient(P1, P2, Q2)
    o3, o4 = _orient(Q1, Q2, P1), _orient(Q1, Q2, P2)
    cross = (o1 * o2 < 0) & (o3 * o4 < 0)  # proper crossings only; collinear touches ignored
    return int(cross.sum())


def crossing_ratio(X: np.ndarray, E: np.ndarray, pairs=None) -> float:
    if pairs is None:
        pairs = nonadjacent_edge_pairs(E)
    cmax = len(pairs[0])
    return crossing_count(X, E, pairs) / cmax if cmax > 0 else 0.0


@dataclass
class GraphCache:
    """Pre-computed arrays for one graph, reused across many drawings."""
    G: nx.Graph
    E: np.ndarray = field(init=False)
    D: np.ndarray = field(init=False)
    iu: np.ndarray = field(init=False)
    ju: np.ndarray = field(init=False)
    pairs: tuple = field(init=False)

    def __post_init__(self):
        self.E, self.D, self.iu, self.ju = graph_arrays(self.G)
        self.pairs = nonadjacent_edge_pairs(self.E)

    @property
    def n(self):
        return self.G.number_of_nodes()


def realism_metrics(X: np.ndarray) -> dict:
    """Shape diagnostics used to reject degenerate drawings (not optimisation targets).
    aspect   = short side / long side of the bounding box (1 = square)
    nn_max   = largest nearest-neighbour distance * sqrt(n)   (large = isolated outlier node)
    nn_min   = smallest pairwise distance * sqrt(n)            (small = overlapping nodes)
    dense    = fraction of nodes inside the densest cell of a 4x4 grid (large = clump)"""
    X = normalise(X)
    n = len(X)
    span = X.max(0) - X.min(0)
    aspect = float(span.min() / span.max()) if span.max() > 0 else 0.0
    Dm = np.linalg.norm(X[:, None] - X[None], axis=-1) + np.eye(n) * 10
    nn = Dm.min(1)
    lo, hi = X.min(0), X.max(0)
    idx = np.minimum(((X - lo) / np.where(hi - lo > 0, hi - lo, 1) * 4).astype(int), 3)
    counts = np.zeros((4, 4)); np.add.at(counts, (idx[:, 0], idx[:, 1]), 1)
    return {"aspect": aspect, "nn_max": float(nn.max() * math.sqrt(n)),
            "nn_min": float(nn.min() * math.sqrt(n)), "dense": float(counts.max() / n)}


REALISM_LIMITS = {"aspect_min": 0.70, "nn_max_max": 2.6, "nn_min_min": 0.20, "dense_max": 0.25}


def realism_ok(r: dict, lim: dict = REALISM_LIMITS) -> bool:
    return (r["aspect"] >= lim["aspect_min"] and r["nn_max"] <= lim["nn_max_max"]
            and r["nn_min"] >= lim["nn_min_min"] and r["dense"] <= lim["dense_max"])


def all_metrics(X: np.ndarray, gc: GraphCache, with_realism: bool = False) -> dict:
    X = normalise(X)
    out = {
        "ksm": ksm(X, gc.D, gc.iu, gc.ju),
        "nu": node_uniformity(X),
        "eld": edge_length_deviation(X, gc.E),
        "xr": crossing_ratio(X, gc.E, gc.pairs),
    }
    if with_realism:
        rm = realism_metrics(X)
        out.update(rm)
        out["realistic"] = realism_ok(rm)
    return out


# ----------------------------------------------------------------------------
# 3. Differentiable surrogates (torch)
# ----------------------------------------------------------------------------
def _torch():
    import torch
    return torch


def t_normalise(X):
    """Differentiable uniform-scale normalisation to the unit box."""
    X = X - X.min(0).values
    return X / (X.max() + 1e-9)


def sur_stress(X, d, iu, ju):
    """Linear-scale stress: sqrt(sum (e - a d)^2 / sum e^2), a = <e,d>/<d,d>. Upper bound of KSM."""
    torch = _torch()
    e = torch.linalg.norm(X[iu] - X[ju], dim=1)
    a = (e * d).sum() / (d * d).sum()
    return torch.sqrt(((e - a * d) ** 2).sum() / (e * e).sum() + 1e-12)


def sur_nu(X, n):
    """Soft-histogram node uniformity on the per-axis bounding box grid."""
    torch = _torch()
    g = int(math.ceil(math.sqrt(n)))
    lo = X.min(0).values
    span = (X.max(0).values - lo).clamp_min(1e-9)
    U = (X - lo) / span
    centres = (torch.arange(g, dtype=X.dtype) + 0.5) / g
    cx, cy = torch.meshgrid(centres, centres, indexing="ij")
    C = torch.stack([cx.reshape(-1), cy.reshape(-1)], 1)          # (g^2, 2)
    sigma = 0.5 / g
    d2 = ((U[:, None, :] - C[None, :, :]) ** 2).sum(-1)           # (n, g^2)
    W = torch.softmax(-d2 / (2 * sigma ** 2), dim=1)
    counts = W.sum(0)
    cbar = n / g ** 2
    num = torch.sqrt(((counts - cbar) ** 2).sum() + 1e-12)
    den = math.sqrt((n - cbar) ** 2 + (g ** 2 - 1) * cbar ** 2)
    return num / den


def sur_eld(X, E):
    torch = _torch()
    L = torch.linalg.norm(X[E[:, 0]] - X[E[:, 1]], dim=1)
    return L.std(unbiased=False) / (L.mean() + 1e-12)


def sur_xr(X, E, pa, pb, k=50.0):
    """Smooth crossing ratio. Orientation tests are normalised (sine of angle) so k is scale free."""
    torch = _torch()
    if len(pa) == 0:
        return X.sum() * 0.0
    P1, P2 = X[E[pa, 0]], X[E[pa, 1]]
    Q1, Q2 = X[E[pb, 0]], X[E[pb, 1]]

    def o(p, q, r):
        v1, v2 = q - p, r - p
        cr = v1[:, 0] * v2[:, 1] - v1[:, 1] * v2[:, 0]
        return cr / (torch.linalg.norm(v1, dim=1) * torch.linalg.norm(v2, dim=1) + 1e-9)

    s1 = -o(P1, P2, Q1) * o(P1, P2, Q2)
    s2 = -o(Q1, Q2, P1) * o(Q1, Q2, P2)
    soft = torch.sigmoid(k * s1) * torch.sigmoid(k * s2)
    return soft.sum() / len(pa)


def sur_realism(X, r_max, lam_aspect=1.0):
    """Penalise outliers (nearest-neighbour distance > r_max) and elongated drawings (aspect ratio far from 1).
    Without this, the optimiser reaches high stress or a target NU through degenerate shapes:
    a few far-away nodes stretching the bounding box, or a thin strip."""
    torch = _torch()
    n = X.shape[0]
    D = torch.cdist(X, X) + torch.eye(n, dtype=X.dtype) * 10.0
    nn = D.min(1).values
    outlier = (torch.relu(nn - r_max) ** 2).sum() / (r_max ** 2)
    span = X.max(0).values - X.min(0).values
    aspect = (span[0] - span[1]) ** 2 / (span.max() ** 2 + 1e-12)
    return outlier + lam_aspect * aspect * n


def sur_overlap(X, r_min=0.03):
    torch = _torch()
    n = X.shape[0]
    iu, ju = torch.triu_indices(n, n, offset=1)
    dist = torch.linalg.norm(X[iu] - X[ju], dim=1)
    return (torch.relu(r_min - dist) ** 2).sum()


# ----------------------------------------------------------------------------
# 4. PDSG Stage 1 — optimise one drawing toward target metric values
# ----------------------------------------------------------------------------
DEFAULT_TOL = {"ksm": 0.01, "nu": 0.02, "eld": 0.02, "xr": 0.01}


@dataclass
class PDSGResult:
    X: np.ndarray
    metrics: dict
    converged: bool
    iters: int
    restarts: int
    seconds: float
    errors: dict


def initial_layout(G, low_stress: bool, rng: np.random.Generator):
    n = G.number_of_nodes()
    if low_stress:
        pos = nx.kamada_kawai_layout(G)
        X = np.array([pos[i] for i in range(n)])
        X = normalise(X) + rng.normal(0, 0.02, size=(n, 2))
    else:
        X = rng.uniform(0, 1, size=(n, 2))
    th = rng.uniform(0, 2 * math.pi)
    R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]])
    return normalise(X @ R.T)


def pdsg_optimise(gc: GraphCache, targets: dict, tol: dict | None = None, X0: np.ndarray | None = None,
                  max_steps: int = 2000, max_restarts: int = 5, lr: float = 0.05, check_every: int = 50,
                  k_xr: float = 50.0, lam_overlap: float = 1.0, spacing_c: float = 0.45, r_min: float | None = None,
                  outlier_c: float = 2.0, lam_realism: float = 1.0,
                  seed: int = 0, verbose: bool = False) -> PDSGResult:
    """
    targets: dict with any of keys 'ksm','nu','eld','xr' -> float target, or None/absent = free.
    Loss = sum_p ((sur_p - t~_p)/tol_p)^2 + lam_overlap * overlap.
    Spacing: overlap penalty with r_min = spacing_c / sqrt(n) (unit box) stops the optimiser reaching
    high stress by collapsing nodes into a clump (a degenerate, trivially detectable drawing).
    Realism: nearest-neighbour distance capped at outlier_c / sqrt(n); aspect ratio pulled toward 1.
    Outer correction: every `check_every` steps the surrogate target t~_p is shifted by (target - exact),
    so the optimiser converges on the EXACT metric even though it differentiates a surrogate.
    """
    torch = _torch()
    tol = {**DEFAULT_TOL, **(tol or {})}
    active = {k: v for k, v in targets.items() if v is not None}
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    t0 = time.time()

    E_t = torch.as_tensor(gc.E, dtype=torch.long)
    d_t = torch.as_tensor(gc.D[gc.iu, gc.ju], dtype=torch.float64)
    iu_t = torch.as_tensor(gc.iu, dtype=torch.long)
    ju_t = torch.as_tensor(gc.ju, dtype=torch.long)
    pa_t = torch.as_tensor(gc.pairs[0], dtype=torch.long)
    pb_t = torch.as_tensor(gc.pairs[1], dtype=torch.long)
    n = gc.n
    if r_min is None:
        r_min = spacing_c / math.sqrt(n)
    r_max = outlier_c / math.sqrt(n)

    def surrogates(Xt):
        Xn = t_normalise(Xt)
        out = {}
        if "ksm" in active:
            out["ksm"] = sur_stress(Xn, d_t, iu_t, ju_t)
        if "nu" in active:
            out["nu"] = sur_nu(Xn, n)
        if "eld" in active:
            out["eld"] = sur_eld(Xn, E_t)
        if "xr" in active:
            out["xr"] = sur_xr(Xn, E_t, pa_t, pb_t, k_xr)
        return out, Xn

    if X0 is None:
        X0 = initial_layout(gc.G, low_stress=(active.get("ksm", 0.5) <= 0.55), rng=rng)
    best = None
    total_iters = 0
    X_start = np.array(X0, dtype=np.float64)

    for restart in range(max_restarts + 1):
        Xt = torch.tensor(X_start, dtype=torch.float64, requires_grad=True)
        opt = torch.optim.Adam([Xt], lr=lr, betas=(0.9, 0.99))
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_steps, eta_min=lr * 0.1)
        sur_target = dict(active)
        for step in range(1, max_steps + 1):
            opt.zero_grad()
            sur, Xn = surrogates(Xt)
            loss = sum(((sur[k] - sur_target[k]) / tol[k]) ** 2 for k in sur)
            loss = loss + lam_overlap * sur_overlap(Xn, r_min) / (r_min ** 2) + lam_realism * sur_realism(Xn, r_max)
            loss.backward()
            opt.step()
            sched.step()
            total_iters += 1
            if step % check_every == 0 or step == max_steps:
                Xnp = normalise(Xt.detach().numpy())
                m = all_metrics(Xnp, gc)
                err = {k: m[k] - active[k] for k in active}
                score = sum(abs(err[k]) / tol[k] for k in active)
                if best is None or score < best[0]:
                    best = (score, Xnp.copy(), m, err)
                if all(abs(err[k]) < tol[k] for k in active):
                    return PDSGResult(_random_flip(Xnp, rng), m, True, total_iters, restart, time.time() - t0, err)
                for k in active:  # outer correction on surrogate targets
                    sur_target[k] = sur_target[k] - 0.8 * err[k]
                if verbose and step % 500 == 0:
                    print(restart, step, {k: round(v, 4) for k, v in m.items()})
        X_start = best[1] + rng.normal(0, 0.1, size=best[1].shape)

    _, Xb, m, err = best
    return PDSGResult(_random_flip(Xb, rng), m, False, total_iters, max_restarts, time.time() - t0, err)


def _random_flip(X, rng):
    """Random reflection per axis: preserves every metric (incl. NU), unlike arbitrary rotation."""
    X = X.copy()
    if rng.random() < 0.5:
        X[:, 0] = 1 - X[:, 0]
    if rng.random() < 0.5:
        X[:, 1] = 1 - X[:, 1]
    if rng.random() < 0.5:
        X = X[:, ::-1].copy()
    return X


# ----------------------------------------------------------------------------
# 5. Rendering
# ----------------------------------------------------------------------------
RENDER_DEFAULT = dict(res=768, pad=24, node_r=6.0, edge_w=1.5, node_fill="#333333", edge_col="#000000",
                      bg="#FFFFFF", supersample=4, antialias=True)


def render_png(gc: GraphCache, X: np.ndarray, path: str | None = None, **kw):
    """Draw edges then nodes. Returns a PIL image; saves to `path` if given."""
    from PIL import Image, ImageDraw
    o = {**RENDER_DEFAULT, **kw}
    ss = o["supersample"] if o["antialias"] else 1
    R = o["res"] * ss
    pad = o["pad"] * ss
    Xn = normalise(X)
    P = pad + Xn * (R - 2 * pad)
    P[:, 1] = R - P[:, 1]  # y up
    img = Image.new("RGB", (R, R), o["bg"])
    dr = ImageDraw.Draw(img)
    ew = max(1, int(round(o["edge_w"] * ss)))
    for u, v in gc.E:
        dr.line([tuple(P[u]), tuple(P[v])], fill=o["edge_col"], width=ew)
    r = o["node_r"] * ss
    for x, y in P:
        dr.ellipse([x - r, y - r, x + r, y + r], fill=o["node_fill"])
    if ss > 1:
        img = img.resize((o["res"], o["res"]), Image.LANCZOS)
    if path:
        img.save(path, optimize=True)
    return img


# ----------------------------------------------------------------------------
# 6. Parsing model output
# ----------------------------------------------------------------------------
_ANS_RE = re.compile(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', re.I)


def parse_answer(text: str) -> dict:
    """Return {'answer': '1'|'2'|'same'|None, 'confidence': int|None, 'reason': str|None, 'parse_ok': bool}."""
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = _ANS_RE.search(text)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
    return out


# ----------------------------------------------------------------------------
# 7. Statistics helpers
# ----------------------------------------------------------------------------
def bootstrap_ci(values, groups=None, n_boot=10000, ci=95, seed=0, stat=np.mean):
    """Bootstrap CI of `stat`. If `groups` is given, resample groups (e.g., graph ids) with replacement."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if groups is None:
        idx = rng.integers(0, len(values), size=(n_boot, len(values)))
        boots = np.array([stat(values[i]) for i in idx])
    else:
        groups = np.asarray(groups)
        uniq = np.unique(groups)
        members = {g: np.where(groups == g)[0] for g in uniq}
        boots = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.choice(uniq, size=len(uniq), replace=True)
            boots[b] = stat(values[np.concatenate([members[g] for g in pick])])
    lo, hi = np.percentile(boots, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(stat(values)), float(lo), float(hi)


Writing /kaggle/working/n01_output/pdcp_utils.py


In [4]:
sys.path.insert(0, OUTPUT_DIR)
import pdcp_utils as U
importlib.reload(U)
print("pdcp_utils", U.__version__)
REPORT = {"env": ENV, "utils_version": U.__version__, "tests": {}}
def record(name, ok, **info):
    REPORT["tests"][name] = {"pass": bool(ok), **info}
    print(("PASS " if ok else "FAIL ") + name, info if info else "")

pdcp_utils 0.1.0


## 2. Exact metrics on toy cases

In [5]:
sq = np.array([[0,0],[1,0],[1,1],[0,1]], float)
gc = U.GraphCache(nx.cycle_graph(4)); m = U.all_metrics(sq, gc)
record("C4_square", all(abs(m[k]) < 1e-12 for k in m), metrics=m)
gc = U.GraphCache(nx.complete_graph(4)); m = U.all_metrics(sq, gc)
record("K4_square_one_crossing", abs(m["xr"] - 1/3) < 1e-12, metrics=m)   # 1 crossing out of 3 crossable pairs
gc = U.GraphCache(nx.path_graph(6)); m = U.all_metrics(np.c_[np.arange(6), np.zeros(6)], gc)
record("path_on_line_zero_stress", m["ksm"] < 1e-12 and m["xr"] == 0, metrics=m)
X1 = np.zeros((9, 2)); X1[0] = [1, 1]
record("nu_all_in_one_cell", U.node_uniformity(X1) > 0.85, nu=U.node_uniformity(X1))
g3 = np.array([[i, j] for i in range(3) for j in range(3)], float)
record("nu_grid_uniform", U.node_uniformity(g3) < 1e-12, nu=U.node_uniformity(g3))

PASS C4_square {'metrics': {'ksm': 0.0, 'nu': 0.0, 'eld': 0.0, 'xr': 0.0}}
PASS K4_square_one_crossing {'metrics': {'ksm': 0.1691019787257628, 'nu': 0.0, 'eld': 0.17157287525381, 'xr': 0.3333333333333333}}
PASS path_on_line_zero_stress {'metrics': {'ksm': 9.383097129595137e-17, 'nu': 0.5, 'eld': 2.1499376424746292e-16, 'xr': 0.0}}
PASS nu_all_in_one_cell {'nu': 0.881917103688197}
PASS nu_grid_uniform {'nu': 0.0}


## 3. Kruskal stress vs scikit-learn isotonic regression
KSM must match `sklearn.isotonic.IsotonicRegression` (secondary approach to ties), the convention behind the 0.4–0.8 range of Mooney et al.

In [6]:
from sklearn.isotonic import IsotonicRegression
rng = np.random.default_rng(SEED); maxdiff = 0.0
for fam, n in [("ER", 25), ("ER", 50), ("BA", 50), ("GRID", 49), ("TREE", 50), ("SBM", 100)]:
    gc = U.GraphCache(U.make_graph(fam, n, SEED))
    for t in range(10 if FAST else 30):
        X = U.normalise(rng.uniform(size=(n, 2)))
        d = gc.D[gc.iu, gc.ju]; e = np.linalg.norm(X[gc.iu] - X[gc.ju], axis=1)
        dh = IsotonicRegression().fit_transform(d, e)
        ref = np.sqrt(((e - dh) ** 2).sum() / (e ** 2).sum())
        maxdiff = max(maxdiff, abs(U.ksm(X, gc.D, gc.iu, gc.ju) - ref))
record("ksm_matches_sklearn", maxdiff < 1e-9, max_abs_diff=float(maxdiff))

PASS ksm_matches_sklearn {'max_abs_diff': 5.551115123125783e-17}


## 4. Surrogates track the exact metrics
The optimiser differentiates surrogates and corrects toward the exact value every 50 steps, so surrogates only need to be *monotone* in the exact metric (Spearman ρ), not equal to it.

In [7]:
from scipy.stats import spearmanr
rows = []
rng = np.random.default_rng(SEED + 1)
for fam, n in [("ER", 25), ("ER", 50), ("BA", 50), ("SBM", 50), ("TREE", 50), ("GRID", 49), ("ER", 100)]:
    gc = U.GraphCache(U.make_graph(fam, n, SEED))
    pos = nx.kamada_kawai_layout(gc.G); P = U.normalise(np.array([pos[i] for i in range(n)]))
    E_t = torch.as_tensor(gc.E); d_t = torch.as_tensor(gc.D[gc.iu, gc.ju]); iu_t = torch.as_tensor(gc.iu); ju_t = torch.as_tensor(gc.ju)
    pa_t, pb_t = torch.as_tensor(gc.pairs[0]), torch.as_tensor(gc.pairs[1])
    for t in range(15 if FAST else 40):
        a = rng.uniform(0, 1)  # interpolate KK layout -> random layout, plus random squeeze
        X = (1 - a) * P + a * rng.uniform(size=(n, 2))
        X = X * np.array([1.0, rng.uniform(0.6, 1.0)])
        X = U.normalise(X); ex = U.all_metrics(X, gc)
        Xt = U.t_normalise(torch.tensor(X))
        rows.append({"family": fam, "n": n, **{"ex_" + k: v for k, v in ex.items()},
                     "su_ksm": float(U.sur_stress(Xt, d_t, iu_t, ju_t)), "su_nu": float(U.sur_nu(Xt, n)),
                     "su_eld": float(U.sur_eld(Xt, E_t)), "su_xr": float(U.sur_xr(Xt, E_t, pa_t, pb_t))})
S = pd.DataFrame(rows)
rho = {k: float(spearmanr(S["ex_" + k], S["su_" + k])[0]) for k in ["ksm", "nu", "eld", "xr"]}
# KSM surrogate uses a linear fit (exact uses isotonic) and NU a soft histogram, so lower thresholds for those two;
# the optimiser's outer correction makes the final EXACT values hit their targets regardless.
THR = {"ksm": 0.80, "nu": 0.85, "eld": 0.95, "xr": 0.95}
record("surrogates_monotone", all(rho[k] > THR[k] for k in rho), spearman=rho, thresholds=THR)
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
for i, k in enumerate(["ksm", "nu", "eld", "xr"]):
    ax[i].scatter(S["ex_" + k], S["su_" + k], s=6, alpha=.5); ax[i].set_xlabel("exact " + k); ax[i].set_ylabel("surrogate")
    ax[i].set_title(f"{k.upper()}  Spearman ρ = {rho[k]:.3f}")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_surrogate_vs_exact.png", dpi=130); plt.show()

PASS surrogates_monotone {'spearman': {'ksm': 0.832936462381991, 'nu': 0.8894412131128225, 'eld': 1.0, 'xr': 0.9992137699895528}, 'thresholds': {'ksm': 0.8, 'nu': 0.85, 'eld': 0.95, 'xr': 0.95}}


## 5. Gradient checks (float64 `torch.autograd.gradcheck`)

In [8]:
from torch.autograd import gradcheck
gc = U.GraphCache(U.make_graph("ER", 10, SEED))
E_t = torch.as_tensor(gc.E); d_t = torch.as_tensor(gc.D[gc.iu, gc.ju], dtype=torch.float64)
iu_t = torch.as_tensor(gc.iu); ju_t = torch.as_tensor(gc.ju)
pa_t, pb_t = torch.as_tensor(gc.pairs[0]), torch.as_tensor(gc.pairs[1])
X0 = torch.tensor(np.random.default_rng(3).uniform(size=(10, 2)), dtype=torch.float64, requires_grad=True)
gres = {}
for name, f in {"stress": lambda X: U.sur_stress(X, d_t, iu_t, ju_t),
                "nu": lambda X: U.sur_nu(X, 10),
                "eld": lambda X: U.sur_eld(X, E_t),
                "xr": lambda X: U.sur_xr(X, E_t, pa_t, pb_t, k=5.0),
                "realism": lambda X: U.sur_realism(X, 2.0 / np.sqrt(10))}.items():
    try:
        gres[name] = bool(gradcheck(f, (X0,), eps=1e-6, atol=1e-4))
    except Exception as ex:
        gres[name] = False; print(name, ex)
record("gradcheck", all(gres.values()), results=gres)

PASS gradcheck {'results': {'stress': True, 'nu': True, 'eld': True, 'xr': True, 'realism': True}}


## 6. Optimiser convergence sweep
Targets inside the natural range; each run uses default settings (2,000 steps, 5 restarts). Records convergence, time, realism.

In [9]:
fams = {"ER": [10, 25, 50, 100], "BA": [25, 50, 100], "SBM": [25, 50, 100], "WS": [25, 50],
        "GRID": [16, 25, 49, 100], "TREE": [25, 50, 100]}
if FAST: fams = {"ER": [25, 50], "TREE": [50], "GRID": [49]}
target_sets = [
    {"ksm": 0.40}, {"ksm": 0.45},
    {"ksm": 0.40, "nu": "nat", "eld": "nat", "xr": "nat"},          # all three proxies held at natural values
    {"ksm": 0.45, "nu": "nat+0.10"},                                   # proxy shifted, stress fixed (counterfactual-style)
    {"ksm": 0.40, "eld": "nat+0.15"},
    {"ksm": 0.40, "xr": "nat+0.03"},
]
conv = []
for fam, sizes in fams.items():
    for n in sizes:
        for gi in range(1 if FAST else 2):
            gc = U.GraphCache(U.make_graph(fam, n, SEED + gi))
            pos = nx.kamada_kawai_layout(gc.G); nat = U.all_metrics(np.array([pos[i] for i in range(n)]), gc)
            for ts in target_sets:
                tg = {}
                for k, v in ts.items():
                    if v == "nat": tg[k] = nat[k]
                    elif isinstance(v, str): tg[k] = nat[k] + float(v.split("+")[1])
                    else: tg[k] = v
                if tg["ksm"] < nat["ksm"] + 0.02:      # below what this graph can reach naturally
                    tg["ksm"] = round(nat["ksm"] + 0.05, 3)
                r = U.pdsg_optimise(gc, tg, seed=SEED + gi)
                rm = U.realism_metrics(r.X)
                conv.append({"family": fam, "n": n, "graph": gi, "targets": json.dumps({k: round(v, 3) for k, v in tg.items()}),
                             "converged": r.converged, "realistic": U.realism_ok(rm), "iters": r.iters,
                             "restarts": r.restarts, "seconds": round(r.seconds, 2),
                             **{k: round(v, 4) for k, v in r.metrics.items()}, **{k: round(v, 3) for k, v in rm.items()}})
        print(fam, "done", f"{time.time() - T_START:.0f}s")
CV = pd.DataFrame(conv); CV.to_csv(f"{OUTPUT_DIR}/convergence.csv", index=False)
summary = CV.groupby("family")[["converged", "realistic", "seconds"]].mean().round(3)
print(summary)
record("convergence_rate", CV.converged.mean() >= 0.90, overall=float(CV.converged.mean()),
       realistic=float(CV.realistic.mean()), median_seconds=float(CV.seconds.median()),
       by_family=summary.to_dict())

ER done 18s
ER done 28s
ER done 37s
ER done 48s
BA done 55s
BA done 61s
BA done 73s
SBM done 84s
SBM done 93s
SBM done 106s
WS done 115s
WS done 158s
GRID done 167s
GRID done 220s
GRID done 376s
GRID done 721s
TREE done 739s
TREE done 758s
TREE done 789s
        converged  realistic  seconds
family                               
BA          1.000      0.861    0.666
ER          1.000      0.854    0.869
GRID        0.917      0.750   11.730
SBM         1.000      0.861    0.921
TREE        1.000      0.833    1.852
WS          1.000      0.917    2.149
PASS convergence_rate {'overall': 0.9824561403508771, 'realistic': 0.8377192982456141, 'median_seconds': 0.6, 'by_family': {'converged': {'BA': 1.0, 'ER': 1.0, 'GRID': 0.917, 'SBM': 1.0, 'TREE': 1.0, 'WS': 1.0}, 'realistic': {'BA': 0.861, 'ER': 0.854, 'GRID': 0.75, 'SBM': 0.861, 'TREE': 0.833, 'WS': 0.917}, 'seconds': {'BA': 0.666, 'ER': 0.869, 'GRID': 11.73, 'SBM': 0.921, 'TREE': 1.852, 'WS': 2.149}}}


## 7. Stress-range diagnostic (decides the design of N02/N03)
For each family and size: hold node uniformity at its Kamada–Kawai value, then find the **lowest** and **highest** Kruskal stress at which the optimiser still converges *and* the drawing passes all realism checks.
Expected from development runs: natural drawings exist only up to roughly KSM ≈ 0.50; above that, stress is reached only by clumps, strips or outliers.

In [10]:
grid = np.round(np.arange(0.10, 0.81, 0.05), 2)
rng_rows = []; gallery = {}
fam_sizes = [("ER", 25), ("ER", 50), ("ER", 100), ("BA", 50), ("SBM", 50), ("WS", 50), ("TREE", 50), ("GRID", 49)]
if FAST: fam_sizes = fam_sizes[:3]
for fam, n in fam_sizes:
    for gi in range(1 if FAST else 3):
        gc = U.GraphCache(U.make_graph(fam, n, SEED + 10 + gi))
        pos = nx.kamada_kawai_layout(gc.G); nat = U.all_metrics(np.array([pos[i] for i in range(n)]), gc)
        ok = {}
        for t in grid:
            r = U.pdsg_optimise(gc, {"ksm": float(t), "nu": nat["nu"]}, seed=SEED, max_steps=1500, max_restarts=2)
            real = U.realism_ok(U.realism_metrics(r.X))
            ok[float(t)] = (r.converged, real)
            if gi == 0 and fam in ("ER", "TREE") and n in (50,) and float(t) in (0.3, 0.4, 0.5, 0.6, 0.7):
                gallery[(fam, float(t))] = (gc, r.X, r.converged and real)
        reach = [t for t, (c, rl) in ok.items() if c]
        natural = [t for t, (c, rl) in ok.items() if c and rl]
        rng_rows.append({"family": fam, "n": n, "graph": gi, "kk_ksm": round(nat["ksm"], 3), "kk_nu": round(nat["nu"], 3),
                         "reach_min": min(reach) if reach else None, "reach_max": max(reach) if reach else None,
                         "natural_min": min(natural) if natural else None, "natural_max": max(natural) if natural else None,
                         "natural_levels": json.dumps(natural)})
    print(fam, n, rng_rows[-1]["natural_min"], "-", rng_rows[-1]["natural_max"], f"{time.time() - T_START:.0f}s")
SR = pd.DataFrame(rng_rows); SR.to_csv(f"{OUTPUT_DIR}/stress_range.csv", index=False)
sr_sum = SR.groupby(["family", "n"])[["kk_ksm", "natural_min", "natural_max", "reach_max"]].agg(["min", "median", "max"])
print(sr_sum)
REPORT["stress_range"] = {f"{f}{n}": {"natural_min_median": float(g.natural_min.median()), "natural_max_median": float(g.natural_max.median())}
                          for (f, n), g in SR.groupby(["family", "n"])}
record("stress_range_measured", True, overall_natural_max_median=float(SR.natural_max.median()))

ER 25 0.3 - 0.5 958s
ER 50 0.3 - 0.55 1183s
ER 100 0.3 - 0.6 1505s
BA 50 0.3 - 0.5 1761s
SBM 50 0.25 - 0.5 1951s
WS 50 0.3 - 0.5 2172s
TREE 50 0.15 - 0.6 2242s
GRID 49 0.1 - 0.5 2340s
           kk_ksm               natural_min              natural_max         \
              min median    max         min median   max         min median   
family n                                                                      
BA     50   0.306  0.309  0.316        0.30   0.30  0.30        0.50   0.50   
ER     25   0.223  0.230  0.302        0.25   0.25  0.30        0.50   0.50   
       50   0.224  0.276  0.280        0.25   0.30  0.30        0.50   0.55   
       100  0.302  0.307  0.317        0.30   0.30  0.30        0.55   0.60   
GRID   49   0.095  0.095  0.095        0.10   0.10  0.10        0.50   0.50   
SBM    50   0.212  0.226  0.232        0.20   0.25  0.25        0.50   0.50   
TREE   50   0.121  0.146  0.147        0.10   0.15  0.15        0.50   0.60   
WS     50   0.257  0.273  

In [11]:
# Gallery: same graph at increasing stress, node uniformity fixed; red border = fails realism checks
if gallery:
    fams_g = sorted({f for f, _ in gallery}); levels = sorted({t for _, t in gallery})
    fig, ax = plt.subplots(len(fams_g), len(levels), figsize=(3 * len(levels), 3 * len(fams_g)))
    ax = np.atleast_2d(ax)
    for i, f in enumerate(fams_g):
        for j, t in enumerate(levels):
            a = ax[i, j]; a.axis("off")
            if (f, t) in gallery:
                gc, X, good = gallery[(f, t)]
                a.imshow(U.render_png(gc, X, res=300, node_r=4)); a.set_title(f"{f}50  KSM {t:.1f}", color="black" if good else "red")
                if not good:
                    for s in a.spines.values(): s.set_visible(True); s.set_color("red"); s.set_linewidth(3)
                    a.axis("on"); a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_stress_gallery.png", dpi=110); plt.show()

## 8. Rendering, parser and bootstrap checks

In [12]:
gc = U.GraphCache(U.make_graph("ER", 25, SEED))
r = U.pdsg_optimise(gc, {"ksm": 0.40}, seed=SEED)
for res in (384, 768):
    img = U.render_png(gc, r.X, f"{OUTPUT_DIR}/fig_sample_render_{res}.png", res=res)
record("render", img.size == (768, 768))
cases = {'{"answer": "1", "confidence": 4, "reason": "even spread"}': "1",
         'Sure! {"answer":"Image 2","confidence":3,"reason":"x"}': "2",
         '```json\n{"answer": "same", "confidence": 2, "reason": "y"}\n```': "same",
         'The answer is: same': "same", '{"answer": "no clear difference"}': "same", 'garbage': None}
pr = {k: U.parse_answer(k)["answer"] == v for k, v in cases.items()}
record("parser", all(pr.values()), cases=list(pr.values()))
v = np.r_[np.ones(70), np.zeros(30)]; g = np.repeat(np.arange(20), 5)
m, lo, hi = U.bootstrap_ci(v, g, n_boot=2000)
record("bootstrap", lo < 0.7 < hi and abs(m - 0.7) < 1e-12, mean=m, ci=[lo, hi])

PASS render 
PASS parser {'cases': [True, True, True, True, True, True]}
PASS bootstrap {'mean': 0.7, 'ci': [0.5, 0.9]}


## 9. Report, cleanup and manifest

In [13]:
REPORT["runtime_min"] = round((time.time() - T_START) / 60, 1)
REPORT["all_pass"] = all(t["pass"] for t in REPORT["tests"].values())
REPORT["realism_limits"] = U.REALISM_LIMITS
REPORT["notes"] = ("KSM uses the secondary (sklearn) tie approach. Surrogates are monotone proxies; exact metrics are "
                   "enforced by outer correction. Realism checks reject clumps, strips, outliers and overlaps. "
                   "Use stress_range.csv to set the per-family KSM range in N02/N03.")
json.dump(REPORT, open(f"{OUTPUT_DIR}/n01_report.json", "w"), indent=2, default=str)
for f in os.listdir(OUTPUT_DIR):
    if f.startswith("__pycache__"):
        import shutil; shutil.rmtree(os.path.join(OUTPUT_DIR, f), ignore_errors=True)
desc = {"pdcp_utils.py": "shared module, import in every later notebook",
        "n01_report.json": "all test results, environment, stress-range summary",
        "convergence.csv": "optimiser convergence sweep (one row per run)",
        "stress_range.csv": "natural stress range per family/size/graph -> sets N02/N03 design",
        "fig_surrogate_vs_exact.png": "surrogate vs exact metric scatter",
        "fig_stress_gallery.png": "same graph at increasing stress; red = degenerate",
        "fig_sample_render_384.png": "sample stimulus 384 px", "fig_sample_render_768.png": "sample stimulus 768 px"}
lines = [f"N01 output  ·  all tests pass: {REPORT['all_pass']}  ·  runtime {REPORT['runtime_min']} min", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(p):
        lines.append(f"{f:32s} {os.path.getsize(p)/1024:9.1f} KB   {desc.get(f, '')}")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines))
print("\n".join(lines)); print()
for k, t in REPORT["tests"].items(): print(("PASS " if t["pass"] else "FAIL ") + k)

N01 output  ·  all tests pass: True  ·  runtime 39.0 min

convergence.csv                       25.1 KB   optimiser convergence sweep (one row per run)
fig_sample_render_384.png             44.5 KB   sample stimulus 384 px
fig_sample_render_768.png             98.7 KB   sample stimulus 768 px
fig_stress_gallery.png               528.1 KB   same graph at increasing stress; red = degenerate
fig_surrogate_vs_exact.png            99.9 KB   surrogate vs exact metric scatter
n01_report.json                        3.7 KB   all test results, environment, stress-range summary
pdcp_utils.py                         21.8 KB   shared module, import in every later notebook
stress_range.csv                       1.9 KB   natural stress range per family/size/graph -> sets N02/N03 design

PASS C4_square
PASS K4_square_one_crossing
PASS path_on_line_zero_stress
PASS nu_all_in_one_cell
PASS nu_grid_uniform
PASS ksm_matches_sklearn
PASS surrogates_monotone
PASS gradcheck
PASS convergence_rate
PASS stress_